# Experiment 7h — Post-hoc calibration (bias + clipping) on the validation predictions

No training. Pure post-processing on the already-shipped predictions, motivated by the 7g diagnostic finding: **73.7% of OOF MSE comes from the top 5% of |residual|**, and every model has a systematic positive residual mean of +0.4 (universal under-prediction).

Two cheap corrections to test, fit on OOF, applied to val:

- **Bias correction** — global or per-series. Removes the +0.4 average under-prediction. Free MSE win from the bias² term in the bias-variance decomposition.
- **Per-series clipping** — cap each prediction at the *p_x* of its series's target distribution (from OOF target as a proxy since `train.csv` may not be available locally). Attacks the catastrophic-over-prediction tail that drives 74% of MSE.

Decision rule (pre-registered): a correction is accepted only if it **reduces OOF MSE by ≥ 0.05** *and* **does not increase OOF WAPE by more than 0.05**. Both thresholds match the 0.05 WAPE noise floor we've used throughout the project.

Outputs (only written if at least one correction passes):

- `validation_predictions_7h_gated_calibrated.csv` — corrected version of the 7f pure-DL submission.
- `validation_predictions_7h_ensemble_calibrated.csv` — corrected version of the 7c ensemble submission.

Both are uploadable to the leaderboard; the gated-calibrated one is the pure-DL artifact we'd actually submit if the diagnostic is right.

---
## §0. Setup & inputs

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Auto-locate the OOF + val CSVs.
_CANDIDATES = [Path("."), Path("experiment-7"),
               Path("/kaggle/input/experiment-7-oof"), Path("/kaggle/working")]
DATA_DIR = next(
    (p for p in _CANDIDATES
     if (p / "oof_predictions_7e_mlp_gated.csv").exists()
     and (p / "oof_predictions_7b_blend.csv").exists()),
    None,
)
assert DATA_DIR is not None, "Could not locate OOF CSVs."
OUTPUT_DIR = DATA_DIR

# Val predictions — may or may not be available depending on where you run this.
VAL_GATED_PATH    = DATA_DIR / "validation_predictions_7f.csv"
VAL_ENSEMBLE_PATH = DATA_DIR / "validation_predictions_7c.csv"
have_val_gated    = VAL_GATED_PATH.exists()
have_val_ensemble = VAL_ENSEMBLE_PATH.exists()

print(f"DATA_DIR = {DATA_DIR.resolve()}")
print(f"val_gated CSV    : {'found' if have_val_gated    else 'NOT FOUND'}")
print(f"val_ensemble CSV : {'found' if have_val_ensemble else 'NOT FOUND'}")
print("  (calibrated submission is only written for files that are found)")

# Pre-registered decision thresholds — set before any numbers below are computed.
MIN_MSE_GAIN_TO_ACCEPT  = 0.05  # MSE units. Below this, change is noise.
MAX_WAPE_LOSS_ALLOWED   = 0.05  # WAPE points. Above this, we sacrificed too much L1.

# Calibration grid
BIAS_VARIANTS  = ["none", "global", "per_series"]
CLIP_VARIANTS  = ["none", "p99", "p99.5", "p99.9"]

DATA_DIR = /home/kshitij-parashar/Downloads/projects/bonus_project/experiment-7
val_gated CSV    : found
val_ensemble CSV : found
  (calibrated submission is only written for files that are found)


---
## §1. Load OOFs and verify alignment

Use the **7e gated-MLP OOF** as the primary fitting set (it's the model whose val submission we're calibrating). Also load the **7b blend OOF** so we can apply the same correction recipe to the ensemble for comparison.

In [2]:
df_gated = pd.read_csv(DATA_DIR / "oof_predictions_7e_mlp_gated.csv", parse_dates=["timestamp"])
df_blend = pd.read_csv(DATA_DIR / "oof_predictions_7b_blend.csv", parse_dates=["timestamp"])

# Merge — keep pred_mlp_gated (7e) and pred_blend (7c-style ensemble) per row, plus shared target.
df = df_gated.merge(
    df_blend[["series_id", "timestamp", "fold", "pred_blend"]],
    on=["series_id", "timestamp", "fold"], how="inner",
)
assert len(df) == len(df_gated) == len(df_blend), \
    f"OOF row counts diverged ({len(df_gated)} vs {len(df_blend)} vs {len(df)})"
print(f"Merged OOF: {len(df):,} rows  ({df.series_id.nunique()} series × {df.fold.nunique()} folds)")

# Sanity check: the residual mean from 7g.
for col in ["pred_mlp_gated", "pred_blend"]:
    resid = df["target"] - df[col]
    print(f"  {col}: mean residual = {resid.mean():+.3f}, OOF MSE = {(resid**2).mean():.4f}, "
          f"OOF WAPE = {100 * resid.abs().sum() / df.target.abs().sum():.4f}")

Merged OOF: 96,768 rows  (96 series × 3 folds)
  pred_mlp_gated: mean residual = +0.406, OOF MSE = 8.7478, OOF WAPE = 14.5424
  pred_blend: mean residual = +0.485, OOF MSE = 8.7571, OOF WAPE = 14.4190


---
## §2. Compute per-series calibration parameters

From OOF only — no leakage from val.

- **Global bias**: `mean(target − pred)` across all rows.
- **Per-series bias**: `mean(target − pred)` within each series.
- **Per-series clipping thresholds**: `target.quantile(q)` per series, where `q ∈ {0.99, 0.995, 0.999}`.

Each series has 1008 OOF rows (3 folds × 336 hours), more than enough for stable p99 estimation.

In [3]:
def per_series_bias(df_, pred_col):
    return df_.assign(resid=lambda d: d["target"] - d[pred_col]) \
              .groupby("series_id")["resid"].mean()

per_series_target_q = (
    df.groupby("series_id")["target"]
      .agg([("p99",   lambda x: np.quantile(x, 0.99)),
            ("p99.5", lambda x: np.quantile(x, 0.995)),
            ("p99.9", lambda x: np.quantile(x, 0.999))])
)

bias_gated_global    = float((df["target"] - df["pred_mlp_gated"]).mean())
bias_blend_global    = float((df["target"] - df["pred_blend"]).mean())
bias_gated_per_sid   = per_series_bias(df, "pred_mlp_gated")
bias_blend_per_sid   = per_series_bias(df, "pred_blend")

print(f"Global bias (gated)   : {bias_gated_global:+.4f}")
print(f"Global bias (ensemble): {bias_blend_global:+.4f}")
print(f"\nPer-series bias (gated) distribution:")
print(f"  mean = {bias_gated_per_sid.mean():+.3f}, std = {bias_gated_per_sid.std():.3f}, "
      f"min = {bias_gated_per_sid.min():+.3f}, max = {bias_gated_per_sid.max():+.3f}")
print(f"\nPer-series clipping thresholds — p99 distribution across series:")
print(f"  mean = {per_series_target_q['p99'].mean():.2f}, "
      f"min = {per_series_target_q['p99'].min():.2f}, "
      f"max = {per_series_target_q['p99'].max():.2f}")
print(f"  (a prediction clipped at p99 stays as-is for ≤1% of train rows in that series)")

Global bias (gated)   : +0.4056
Global bias (ensemble): +0.4852

Per-series bias (gated) distribution:
  mean = +0.406, std = 0.322, min = -0.434, max = +1.174

Per-series clipping thresholds — p99 distribution across series:
  mean = 25.87, min = 16.36, max = 34.70
  (a prediction clipped at p99 stays as-is for ≤1% of train rows in that series)


---
## §3. Evaluate the grid on OOF

12 combinations (3 bias × 4 clipping). For each, compute OOF MSE, RMSE, MAE, WAPE for both the gated MLP and the ensemble blend. Report deltas vs the no-correction baseline.

In [4]:
def metrics(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    err = y - p
    return dict(
        MAE=float(np.mean(np.abs(err))),
        MSE=float(np.mean(err ** 2)),
        RMSE=float(np.sqrt(np.mean(err ** 2))),
        WAPE=float(100 * np.sum(np.abs(err)) / np.sum(np.abs(y))),
    )

def apply_correction(df_, pred_col, bias_kind, clip_kind,
                     bias_global, bias_per_sid, q_lookup):
    p = df_[pred_col].to_numpy().copy()
    if bias_kind == "global":
        p = p + bias_global
    elif bias_kind == "per_series":
        p = p + df_["series_id"].map(bias_per_sid).to_numpy()
    if clip_kind != "none":
        caps = df_["series_id"].map(q_lookup[clip_kind]).to_numpy()
        p = np.minimum(p, caps)
    # Always clip at zero (target is non-negative)
    p = np.clip(p, 0.0, None)
    return p

def grid_eval(df_, pred_col, bias_global, bias_per_sid, q_lookup):
    rows = []
    base = metrics(df_["target"], df_[pred_col])
    for bk in BIAS_VARIANTS:
        for ck in CLIP_VARIANTS:
            corrected = apply_correction(df_, pred_col, bk, ck,
                                         bias_global, bias_per_sid, q_lookup)
            m = metrics(df_["target"], corrected)
            rows.append({
                "bias": bk, "clip": ck,
                **m,
                "ΔMSE":  m["MSE"]  - base["MSE"],
                "ΔWAPE": m["WAPE"] - base["WAPE"],
            })
    return pd.DataFrame(rows), base

q_lookup = {k: per_series_target_q[k] for k in ["p99", "p99.5", "p99.9"]}

print("=== Gated MLP (7e) — calibration grid on OOF ===\n")
grid_gated, base_gated = grid_eval(df, "pred_mlp_gated",
                                   bias_gated_global, bias_gated_per_sid, q_lookup)
print(grid_gated.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  Baseline: MAE={base_gated['MAE']:.4f}  MSE={base_gated['MSE']:.4f}  "
      f"RMSE={base_gated['RMSE']:.4f}  WAPE={base_gated['WAPE']:.4f}")

print("\n=== Ensemble blend (7c) — calibration grid on OOF ===\n")
grid_blend, base_blend = grid_eval(df, "pred_blend",
                                   bias_blend_global, bias_blend_per_sid, q_lookup)
print(grid_blend.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  Baseline: MAE={base_blend['MAE']:.4f}  MSE={base_blend['MSE']:.4f}  "
      f"RMSE={base_blend['RMSE']:.4f}  WAPE={base_blend['WAPE']:.4f}")

=== Gated MLP (7e) — calibration grid on OOF ===

      bias  clip    MAE    MSE   RMSE    WAPE    ΔMSE   ΔWAPE
      none  none 1.5349 8.7478 2.9577 14.5424  0.0000  0.0000
      none   p99 1.5369 8.7657 2.9607 14.5619  0.0180  0.0195
      none p99.5 1.5356 8.7536 2.9587 14.5492  0.0059  0.0069
      none p99.9 1.5348 8.7476 2.9576 14.5421 -0.0002 -0.0003
    global  none 1.5865 8.5832 2.9297 15.0317 -0.1645  0.4893
    global   p99 1.5886 8.6016 2.9328 15.0518 -0.1462  0.5094
    global p99.5 1.5872 8.5893 2.9308 15.0386 -0.1584  0.4962
    global p99.9 1.5865 8.5829 2.9297 15.0312 -0.1649  0.4888
per_series  none 1.5976 8.4804 2.9121 15.1368 -0.2673  0.5944
per_series   p99 1.5998 8.4998 2.9154 15.1574 -0.2480  0.6150
per_series p99.5 1.5984 8.4867 2.9132 15.1443 -0.2611  0.6019
per_series p99.9 1.5976 8.4802 2.9121 15.1366 -0.2675  0.5942

  Baseline: MAE=1.5349  MSE=8.7478  RMSE=2.9577  WAPE=14.5424

=== Ensemble blend (7c) — calibration grid on OOF ===

      bias  clip    MAE  

---
## §4. Pick the winning correction per model

Apply the pre-registered rule: keep the configuration with the **lowest MSE** *subject to* ΔWAPE ≤ `MAX_WAPE_LOSS_ALLOWED`. If no configuration beats the no-correction baseline by `MIN_MSE_GAIN_TO_ACCEPT` MSE, fall back to no correction (do not submit a calibrated version).

In [5]:
def pick_winner(grid, base, label):
    feasible = grid[grid["ΔWAPE"] <= MAX_WAPE_LOSS_ALLOWED].copy()
    if feasible.empty:
        print(f"  {label}: NO feasible correction (every option exceeded the {MAX_WAPE_LOSS_ALLOWED} WAPE budget). Keeping baseline.")
        return None
    winner = feasible.sort_values("MSE").iloc[0]
    gain = base["MSE"] - winner["MSE"]
    print(f"  {label}: best feasible = bias={winner['bias']:>10}, clip={winner['clip']:>6}  →  "
          f"MSE {winner['MSE']:.4f} (Δ {-gain:+.4f}), WAPE {winner['WAPE']:.4f} (Δ {winner['ΔWAPE']:+.4f})")
    if gain < MIN_MSE_GAIN_TO_ACCEPT:
        print(f"    ✗ MSE gain {gain:.4f} < {MIN_MSE_GAIN_TO_ACCEPT} threshold — DO NOT calibrate. Ship baseline.")
        return None
    print(f"    ✅ MSE gain {gain:.4f} ≥ {MIN_MSE_GAIN_TO_ACCEPT} — calibrate val predictions with this recipe.")
    return dict(bias=winner["bias"], clip=winner["clip"], expected_mse=winner["MSE"], expected_wape=winner["WAPE"])

print("=== Winner selection ===\n")
winner_gated = pick_winner(grid_gated, base_gated, "Gated MLP (7e)")
winner_blend = pick_winner(grid_blend, base_blend, "Ensemble blend (7c)")

=== Winner selection ===

  Gated MLP (7e): best feasible = bias=      none, clip= p99.9  →  MSE 8.7476 (Δ -0.0002), WAPE 14.5421 (Δ -0.0003)
    ✗ MSE gain 0.0002 < 0.05 threshold — DO NOT calibrate. Ship baseline.
  Ensemble blend (7c): best feasible = bias=      none, clip= p99.9  →  MSE 8.7571 (Δ -0.0001), WAPE 14.4190 (Δ +0.0000)
    ✗ MSE gain 0.0001 < 0.05 threshold — DO NOT calibrate. Ship baseline.


---
## §5. Apply winners to val predictions and write calibrated submissions

If §4 picked a winner, apply the same recipe (bias from OOF, clip thresholds from OOF target) to the val predictions and write the calibrated submission CSV. If no winner, this section is a no-op.

In [ ]:
def apply_to_val(val_df, bias_kind, clip_kind, bias_global, bias_per_sid, q_lookup):
    p = val_df["prediction"].to_numpy().copy()
    if bias_kind == "global":
        p = p + bias_global
    elif bias_kind == "per_series":
        p = p + val_df["series_id"].map(bias_per_sid).fillna(0.0).to_numpy()
    if clip_kind != "none":
        caps = val_df["series_id"].map(q_lookup[clip_kind]).to_numpy()
        # If a val series_id has no OOF cap (shouldn't happen, but be safe), don't clip that row.
        caps = np.where(np.isnan(caps), np.inf, caps)
        p = np.minimum(p, caps)
    p = np.clip(p, 0.0, None)
    return p

wrote = []

if winner_gated is not None and have_val_gated:
    val_g = pd.read_csv(VAL_GATED_PATH, parse_dates=["timestamp"])
    val_g["prediction"] = apply_to_val(val_g, winner_gated["bias"], winner_gated["clip"],
                                       bias_gated_global, bias_gated_per_sid, q_lookup)
    out_path = OUTPUT_DIR / "validation_predictions_7h_gated_calibrated.csv"
    val_g[["series_id", "timestamp", "prediction"]].to_csv(out_path, index=False)
    wrote.append(out_path)
    print(f"✅ Wrote {len(val_g):,} calibrated gated predictions → {out_path}")
    print(f"   recipe: bias={winner_gated['bias']}, clip={winner_gated['clip']}")
    print(f"   expected OOF MSE = {winner_gated['expected_mse']:.4f}, WAPE = {winner_gated['expected_wape']:.4f}")
elif winner_gated is None:
    print("~  Skipping gated submission — no calibration passed §4 thresholds.")
else:
    print("~  Skipping gated submission — val CSV not found.")

if winner_blend is not None and have_val_ensemble:
    val_b = pd.read_csv(VAL_ENSEMBLE_PATH, parse_dates=["timestamp"])
    val_b["prediction"] = apply_to_val(val_b, winner_blend["bias"], winner_blend["clip"],
                                       bias_blend_global, bias_blend_per_sid, q_lookup)
    out_path = OUTPUT_DIR / "validation_predictions_7h_ensemble_calibrated.csv"
    val_b[["series_id", "timestamp", "prediction"]].to_csv(out_path, index=False)
    wrote.append(out_path)
    print(f"✅ Wrote {len(val_b):,} calibrated ensemble predictions → {out_path}")
    print(f"   recipe: bias={winner_blend['bias']}, clip={winner_blend['clip']}")
    print(f"   expected OOF MSE = {winner_blend['expected_mse']:.4f}, WAPE = {winner_blend['expected_wape']:.4f}")
elif winner_blend is None:
    print("~  Skipping ensemble submission — no calibration passed §4 thresholds.")
else:
    print("~  Skipping ensemble submission — val CSV not found.")

if not wrote:
    print("\nNo submissions written. Either no correction was useful, or val CSVs weren't available.")
else:
    print(f"\nTotal calibrated submissions written: {len(wrote)}")

---
## §6. Interpreting the leaderboard result after upload

The 7g diagnostic predicts: a meaningful chunk of MSE comes from a few extreme over-predictions, and calibration should attack them. Specifically:

- **If gated-MLP LB MSE drops from 9.37 toward ~8.5 *without* WAPE rising past 14.85**: calibration works as predicted. Pure-DL artifact gets stronger on Overall rank.
- **If MSE drops AND WAPE also drops slightly**: the systematic +0.4 bias was a free win on both axes.
- **If MSE barely moves**: the val MSE distribution is different from OOF (the catastrophic predictions on val aren't matched by OOF outliers). Calibration is dataset-coupled and our OOF didn't sample the right outliers.
- **If MSE drops but WAPE rises noticeably (> 0.1)**: clipping was too aggressive. Re-run with `MAX_WAPE_LOSS_ALLOWED = 0.02` or skip the clip half of the recipe.

Either way, this is the cheapest possible attempt at Chronos's rank — no retraining, no GPU, fitted entirely from data we already have.